In [ ]:
from pathlib import Path
import json, random, numpy as np, pandas as pd, torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, matthews_corrcoef, roc_auc_score, average_precision_score, classification_report, confusion_matrix
from sklearn.preprocessing import label_binarize
import matplotlib.pyplot as plt
import seaborn as sns
from Model import create_model

DATA_DIR = Path("Data/processed")
MODEL_DIR = Path("Models/DLASS_PRP")
MODEL_DIR.mkdir(parents=True, exist_ok=True)
BATCH_SIZE = 16
EPOCHS = 100
LR = 0.001
WEIGHT_DECAY = 0.0001
NUM_CLASSES = 3
RANDOM_STATE = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_STATE)

train_df = pd.read_csv(DATA_DIR / "train_processed.csv")
test_df = pd.read_csv(DATA_DIR / "test_processed.csv")
with open(DATA_DIR / "selected_features.json") as f:
    selected_features = json.load(f)
with open(DATA_DIR / "target_classes.json") as f:
    target_mapping = json.load(f)

X = train_df[selected_features].values.astype(np.float32)
y = train_df["risk_level"].values.astype(np.int64)
X_test = test_df[selected_features].values.astype(np.float32)
y_test = test_df["risk_level"].values.astype(np.int64)

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.10, stratify=y, random_state=RANDOM_STATE
)

print("Device:", DEVICE)
print("Train:", X_train.shape, "Validation:", X_val.shape, "Test:", X_test.shape)
print("Features:", len(selected_features))
print("Classes:", target_mapping)

In [ ]:
class TabularDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_loader = DataLoader(TabularDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(TabularDataset(X_val, y_val), batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(TabularDataset(X_test, y_test), batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
model = create_model(num_features=len(selected_features), num_classes=NUM_CLASSES).to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=10
)

print(model)
print("Parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))

In [ ]:
def run_epoch(loader, training=False):
    model.train() if training else model.eval()
    total_loss, all_y, all_pred, all_prob = 0, [], [], []

    with torch.set_grad_enabled(training):
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            if training:
                optimizer.zero_grad()

            logits = model(xb)
            loss = criterion(logits, yb)

            if training:
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * len(yb)
            probs = torch.softmax(logits, dim=1)
            all_y.extend(yb.cpu().numpy())
            all_pred.extend(probs.argmax(dim=1).cpu().numpy())
            all_prob.extend(probs.cpu().numpy())

    return total_loss / len(loader.dataset), np.array(all_y), np.array(all_pred), np.array(all_prob)


history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
best_val_loss = float("inf")

for epoch in range(EPOCHS):
    train_loss, ytr, ptr, probtr = run_epoch(train_loader, training=True)
    val_loss, yv, pv, probv = run_epoch(val_loader, training=False)

    train_acc = accuracy_score(ytr, ptr)
    val_acc = accuracy_score(yv, pv)

    scheduler.step(val_loss)
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)

    print(f"Epoch [{epoch+1:03d}/{EPOCHS}] | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | LR: {optimizer.param_groups[0]['lr']:.6f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), MODEL_DIR / "best_model.pth")

model.load_state_dict(torch.load(MODEL_DIR / "best_model.pth", map_location=DEVICE))
print("\nBest model loaded.")

In [ ]:
model.eval()
test_loss, yt, pt, probt = run_epoch(test_loader, training=False)
class_names = [target_mapping[str(i)] for i in range(NUM_CLASSES)]

metrics = {
    "Accuracy": accuracy_score(yt, pt),
    "Precision": precision_score(yt, pt, average="macro", zero_division=0),
    "Recall": recall_score(yt, pt, average="macro", zero_division=0),
    "F1-Score": f1_score(yt, pt, average="macro", zero_division=0),
    "MCC": matthews_corrcoef(yt, pt),
    "AUC": roc_auc_score(yt, probt, multi_class="ovr", average="macro"),
    "AP": average_precision_score(label_binarize(yt, classes=np.arange(NUM_CLASSES)), probt, average="macro")
}

metrics_df = pd.DataFrame([metrics]).T
metrics_df.columns = ["Test Score"]
print("Test Metrics")
display(metrics_df)

print("\nClassification Report")
print(classification_report(yt, pt, target_names=class_names, zero_division=0))

cm = confusion_matrix(yt, pt, labels=np.arange(NUM_CLASSES))
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")
plt.title("DLASS-PRP Confusion Matrix")
plt.tight_layout()
plt.savefig(MODEL_DIR / "confusion_matrix.png", dpi=300)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(history["train_loss"], label="Training Loss")
plt.plot(history["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")
plt.legend()
plt.tight_layout()
plt.savefig(MODEL_DIR / "loss_curve.png", dpi=300)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(history["train_acc"], label="Training Accuracy")
plt.plot(history["val_acc"], label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training and Validation Accuracy")
plt.legend()
plt.tight_layout()
plt.savefig(MODEL_DIR / "accuracy_curve.png", dpi=300)
plt.show()

metrics_df.to_csv(MODEL_DIR / "test_metrics.csv")
pd.DataFrame({
    "Actual": yt,
    "Predicted": pt,
    "Actual_Label": [class_names[i] for i in yt],
    "Predicted_Label": [class_names[i] for i in pt]
}).to_csv(MODEL_DIR / "test_predictions.csv", index=False)

pd.DataFrame(history).to_csv(MODEL_DIR / "training_history.csv", index=False)
print("Results saved to:", MODEL_DIR.resolve())